In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['datasets', 'competitions', 'notebooks']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [2]:
from pathlib import Path

for folder in Path("/kaggle/input").iterdir():
    print(f"\nDataset: {folder.name}")

    for file in folder.rglob("*"):
        if file.is_file() and (
            "baseline" in file.name.lower()
            or "optimal" in file.name.lower()
            or file.suffix.lower() in [".ipynb", ".py"]
        ):
            print("  ", file)


Dataset: datasets
   /kaggle/input/datasets/hannaleysen/gith-l/Kaggle_notebook.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/Test_push_pandas.py
   /kaggle/input/datasets/hannaleysen/gith-l/template.py
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-02-KNN.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-03-KNN+PCA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-04-LDA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-05-linear-regression-newton.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-06-MLP.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-07-MLP+SDG.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/hms-experiment-08-XGBoost-full-mode.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/stap-1-baseline-doen-werken-zonder-errors.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/src/hms_knn.py
   /k

In [3]:
# geoptimaliseerde baseline inlezen 


import nbformat
import ast
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/hannaleysen/basline-optimalisatie/__notebook__.ipynb")
nb = nbformat.read(NOTEBOOK_PATH, as_version=4)

# Haal ALLEEN de functie-definities op uit cel 5 (zodat er geen 'df' wordt verwacht)
tree = ast.parse(nb.cells[5].source)
selected_nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef)]

clean_tree = ast.Module(body=selected_nodes, type_ignores=[])
ast.fix_missing_locations(clean_tree)
exec(compile(clean_tree, "<optimized_features>", "exec"), globals())

print("Geoptimaliseerde featurefuncties veilig geladen!")

print("build_features:", build_features)
print("window_features:", window_features)
print("spectrogram_features:", spectrogram_features)
print("band_columns:", band_columns)

Geoptimaliseerde featurefuncties veilig geladen!
build_features: <function build_features at 0x7849e65fb920>
window_features: <function window_features at 0x7849e65fb7e0>
spectrogram_features: <function spectrogram_features at 0x7849e65fb560>
band_columns: <function band_columns at 0x7849e65fb4c0>


In [4]:
import inspect

print(inspect.signature(build_features))
print(inspect.signature(window_features))
print(inspect.signature(spectrogram_features))

(meta, spec_dir, is_test=False)
(spec, cols)
(path, windows, is_test)


In [5]:
from pathlib import Path
import importlib.util

scripts = list(Path("/kaggle/input").rglob("hms_sklearn_baseline.py"))

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht 1 baseline-bestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]

spec = importlib.util.spec_from_file_location("hms_baseline", SCRIPT_PATH)
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline geladen:", SCRIPT_PATH)

Baseline geladen: /kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [6]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [7]:
import os
import time
import warnings
import ast
import nbformat
import numpy as np
import pandas as pd
import inspect

from pathlib import Path
from joblib import Parallel, delayed

In [8]:
# DATA INLADEN EN GEOPTIMALISEERDE FEATURES
N_JOBS = 2
REGIONS = ["LL", "RL", "LP", "RP"]
BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 20)}


# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification")

assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen (oude baseline blijft hiervoor gebruikt)
df = baseline.load_labels(DATA_DIR, max_specs=None, seed=42)
print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. NIEUW: geoptimaliseerde feature-extractie
X_df = build_features(df, DATA_DIR / "train_spectrograms")

# 3. Controle van het aantal features
feats = [c for c in X_df.columns if c != "eeg_id"]
print("\nAantal geoptimaliseerde features:", len(feats))
assert len(feats) == 192, (f"Verwacht 192 features, maar gevonden: {len(feats)}")

# 4. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id", how="inner")

# 5. Matrix voor XGBoost
X = baseline.to_matrix(df, feats)

# 6. Doelvariabelen en groepen
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)
groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor XGBoost!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

print("\nEerste 20 features:")
for i, feature in enumerate(feats[:20]):
    print(f"{i+1}. {feature}")


(17089, 12) EEGs
1950 verschillende patiënten
11138 spectrogrammen
11138 spectrograms -> 17089 feature rows in 394s

Aantal geoptimaliseerde features: 192

Data klaar voor XGBoost!
Aantal EEG-records: 17089
Aantal features: 192
Aantal klassen: 6
Vorm van X: (17089, 192)
Vorm van Y: (17089, 6)

Eerste 20 features:
1. LL_delta_mean
2. LL_delta_std
3. LL_delta_min
4. LL_delta_max
5. LL_delta_median
6. LL_delta_q25
7. LL_delta_q75
8. LL_delta_iqr
9. LL_delta_trend
10. LL_theta_mean
11. LL_theta_std
12. LL_theta_min
13. LL_theta_max
14. LL_theta_median
15. LL_theta_q25
16. LL_theta_q75
17. LL_theta_iqr
18. LL_theta_trend
19. LL_alpha_mean
20. LL_alpha_std


In [9]:
# Bibliotheken importeren 
import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, ParameterGrid


In [10]:
# Soft labels geschikt maken voor XGBoost 
def expand_soft_labels(X, Y):
    n, k = Y.shape
    X_repeated = np.repeat(X, k, axis=0)
    y_repeated = np.tile(np.arange(k), n)
    weights = Y.reshape(-1)
    keep = weights > 0

    return (X_repeated[keep], y_repeated[keep], weights[keep])


In [11]:
# Bouw XGBoost Model 

XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 6,
    "eval_metric": "mlogloss",

    "tree_method": "hist",

    "max_depth": 3,
    "eta": 0.05,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "min_child_weight": 3,
    "lambda": 5.0,
    "alpha": 0.1,

    "seed": 42,
    "nthread": 2}

NUM_BOOST_ROUND = 500


def train_xgboost(X_train, Y_train, X_valid, Y_valid):
    # 1. Soft labels uitbreiden voor de trainingsdata
    X_train_exp, y_train_exp, train_weights = expand_soft_labels(X_train, Y_train)
    dtrain = xgb.DMatrix(X_train_exp, label=y_train_exp, weight=train_weights)
    # 2. Soft labels uitbreiden voor de validatiedata
    X_valid_exp, y_valid_exp, valid_weights = expand_soft_labels(X_valid, Y_valid)
    dvalid = xgb.DMatrix(X_valid_exp, label=y_valid_exp, weight=valid_weights)
    # 3. XGBoost trainen met early stopping
    model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=1500, evals=[(dtrain, "train"), (dvalid, "valid")], early_stopping_rounds=100, verbose_eval=100)
    return model

def predict_xgboost(model, X_valid):
    dvalid = xgb.DMatrix(X_valid)
    # Als early stopping gebruikt werd:
    if model.attr("best_iteration") is not None:
        predictions = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    # Als early stopping niet gebruikt werd:
    else:
        predictions = model.predict(dvalid)
        
    predictions = predictions.reshape(-1, 6)
    predictions = np.clip(predictions, 1e-15, 1.0)
    predictions /= predictions.sum(axis=1, keepdims=True)

    return predictions

In [12]:
# functie om KL-score te berekenen 
def kl_score(y_true, y_pred, eps=1e-15):
    """
    Bereken de gemiddelde KL-divergentie.
    Een lagere score is beter.
    """

    p = np.clip(y_pred, eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)

    t = np.asarray(y_true)

    terms = np.zeros_like(t, dtype=np.float64)
    mask = t > 0

    terms[mask] = (
        t[mask] * np.log(t[mask] / p[mask])
    )

    return np.mean(np.sum(terms, axis=1))

In [13]:
# Cross-validatie met early stopping

N_FOLDS = 3
xgb_oof = np.zeros_like(Y, dtype=np.float64)
fold_scores = []
best_iterations = []

cv = GroupKFold(n_splits=N_FOLDS)

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, Y, groups), start=1):
    print(f"Start XGBoost fold {fold}/{N_FOLDS}")

    # Train- en validatiedata apart houden
    X_train = X[train_idx]
    Y_train = Y[train_idx]
    X_valid = X[valid_idx]
    Y_valid = Y[valid_idx]

    # Train met early stopping
    model = train_xgboost(X_train, Y_train, X_valid,Y_valid)

    # Beste aantal iteraties opslaan
    best_iterations.append(model.best_iteration + 1)

    # Voorspellingen maken met de beste iteratie
    predictions = predict_xgboost(model, X_valid)
    xgb_oof[valid_idx] = predictions
    score = kl_score(Y_valid, predictions)
    fold_scores.append(score)

    print(f"Fold {fold} KL: {score:.6f}")
    print(f"Beste iteratie: {model.best_iteration + 1}")

print("Gemiddelde fold KL:", np.mean(fold_scores))
print("Totale OOF KL:", kl_score(Y, xgb_oof))
print("Gemiddelde beste iteratie:", np.mean(best_iterations))


# --- Evaluatie ---
hi = num_votes >= 10
print("\nKL per rij, alle rijen      :", kl_score(Y, xgb_oof))
print("KL per rij, >=10 stemmen    :", kl_score(Y[hi], xgb_oof[hi]))


Start XGBoost fold 1/3
[0]	train-mlogloss:1.55552	valid-mlogloss:1.61505
[100]	train-mlogloss:1.09200	valid-mlogloss:1.32459
[200]	train-mlogloss:0.99746	valid-mlogloss:1.29287
[300]	train-mlogloss:0.93883	valid-mlogloss:1.28735
[400]	train-mlogloss:0.89282	valid-mlogloss:1.28495
[498]	train-mlogloss:0.85675	valid-mlogloss:1.28664
Fold 1 KL: 0.949175
Beste iteratie: 399
Start XGBoost fold 2/3
[0]	train-mlogloss:1.57661	valid-mlogloss:1.56718
[100]	train-mlogloss:1.09202	valid-mlogloss:1.28602
[200]	train-mlogloss:0.99165	valid-mlogloss:1.25357
[300]	train-mlogloss:0.93007	valid-mlogloss:1.24452
[400]	train-mlogloss:0.88304	valid-mlogloss:1.23952
[500]	train-mlogloss:0.84392	valid-mlogloss:1.23946
[554]	train-mlogloss:0.82571	valid-mlogloss:1.23977
Fold 2 KL: 0.885058
Beste iteratie: 455
Start XGBoost fold 3/3
[0]	train-mlogloss:1.57698	valid-mlogloss:1.57067
[100]	train-mlogloss:1.11522	valid-mlogloss:1.24314
[200]	train-mlogloss:1.01440	valid-mlogloss:1.20042
[300]	train-mlogloss:0.95

In [14]:
# ===== FINAL XGBoost: fit op ALLE trainingsdata, voorspel test, schrijf submission.csv =====

# 1. Model trainen op alle trainingsdata (X en Y zijn al de volledige set)
final_rounds = int(np.median(best_iterations))
X_exp, y_exp, weights = expand_soft_labels(X, Y)
dtrain = xgb.DMatrix(X_exp, label=y_exp, weight=weights)
final_model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=final_rounds, verbose_eval=False)

# 2. Testfeatures (zelfde stappen als in de baseline)
test = pd.read_csv(DATA_DIR / "test.csv")
T_df = build_features(test, DATA_DIR / "test_spectrograms", is_test=True)
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")
T = baseline.to_matrix(T_df, feats)
assert T.shape[1] == 192


# 3. Voorspellen en submission schrijven
probs = predict_xgboost(final_model, T)

sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[baseline.CLASSES] = probs

assert len(sub) == len(test)
assert not sub.isna().any().any()
assert np.allclose(sub[baseline.CLASSES].sum(axis=1), 1.0)

sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv", sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
Saved submission.csv (1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.060196,0.03691,0.002462,0.278567,0.064699,0.557166
